In [1]:
import numpy as np

In [2]:
# Dot Product (vector)

a1 = np.array([1,2,3])
a2 = np.array([4,5,6])

dot_a1a2 = np.dot(a1,a2) # 1*4 + 2*5 + 3*6 = 4 + 10 + 18 = 32 
print(dot_a1a2) # Scalar

32


In [15]:
# Another example of dot product:

a3 = np.array([1,2])
a4 = np.array([3,4])

# Manual dot product: 1*3 + 2*4 = 3+8 = 11
# Numpy dot product:
dot_a3a4 = np.dot(a3,a4)
print(dot_a3a4)

11


Geometrically, the dot product measures how much two vectors **align**:
- If they point in the same direction, the dot product is positive.
- If they are perpendicular, dot product is zero.
- If they point in opposite directions, the dot product is negative.

In [3]:
# Matrix Multiplication

m1 = np.array([[1,2],
               [3,4]])
m2 = np.array([[5,6],
               [7,8]])

MatMult_m1m2 = np.dot(m1,m2) # or m1 @ m2
# Manual calculation
# MatMutl [0,0]: 1*5 + 2*7 = 5+14 = 19
# MatMutl [0,1]: 1*6 + 2*8 = 6+16 = 22
# MatMutl [1,0]: 3*5 + 4*7 = 15+28 = 43
# MatMutl [1,1]: 3*6 + 4*8 = 18+32 = 50

print(MatMult_m1m2)

[[19 22]
 [43 50]]


In [4]:
# Matrix multiplication combines transformations, not elementwise.

In [5]:
# Solving Linear Systems (Ax = b)

A = np.array([[1,2],
              [4,5]])
b = np.array([6,7])

x = np.linalg.solve(A,b)
print(x)

[-5.33333333  5.66666667]


In [6]:
# Singular Value Decomposition

m3 = np.array([[1,2],
               [3,4],
               [5,6]])
print(m3, "\n")

U, S, Vt = np.linalg.svd(m3, full_matrices=False)
print("U: \n", U)
print("S: \n", S) # Contains singular values. (important for rank and conditioning)
print("Vt: \n", Vt)

# Reconstruct original matrix
m3_reconst = U @ np.diag(S) @ Vt
print("\n", m3_reconst)

[[1 2]
 [3 4]
 [5 6]] 

U: 
 [[-0.2298477   0.88346102]
 [-0.52474482  0.24078249]
 [-0.81964194 -0.40189603]]
S: 
 [9.52551809 0.51430058]
Vt: 
 [[-0.61962948 -0.78489445]
 [-0.78489445  0.61962948]]

 [[1. 2.]
 [3. 4.]
 [5. 6.]]


In [7]:
# Rank & Condition Number

rank = np.linalg.matrix_rank(A)
print(rank)

cond_num = np.linalg.cond(A)
print(cond_num)

2
15.267836167327582


Notes:
- Condition number > 1: large values indicate potential instability; very high = “ill-conditioned” matrix.
- Rank
    - Rank tells us the number of independent columns/rows (how much real information is present).
    - In SVD terms, rank is the number of non-zero singular values.
    - If rank < number of columns:
        - System has infinite solutions
        - Regression has multicollinearity
        - Matrix is not invertible

In [8]:
# Let's do a simple linear regression manually.

# Data
X = np.array([[1, 1],
              [1, 2],
              [1, 3]])  # 1st column = intercept
y = np.array([1, 2, 2])

# Beta = (Xt X)**-1 Xt y
beta = np.linalg.inv(X.T @ X) @ X.T @ y
print("Coefficients: ", beta)

Coefficients:  [0.66666667 0.5       ]


In [9]:
import sys
print(sys.executable)

C:\Users\Erfan\AppData\Local\Programs\Python\Python314\python.exe


In [10]:
# Let's compare it with statsmodels.

import statsmodels.api as sm

model = sm.OLS(y, X).fit()
print(model.summary())

# Coefficients from manual and statsmodels should match closely. Differences appear if the matrix is ill-conditioned.

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.750
Model:                            OLS   Adj. R-squared:                  0.500
Method:                 Least Squares   F-statistic:                     3.000
Date:                Sun, 15 Feb 2026   Prob (F-statistic):              0.333
Time:                        17:19:14   Log-Likelihood:               0.078742
No. Observations:                   3   AIC:                             3.843
Df Residuals:                       1   BIC:                             2.040
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.6667      0.624      1.069      0.4

C:\Users\Erfan\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\stats\stattools.py:74: ValueWarning: omni_normtest is not valid with less than 8 observations; 3 samples were given.
  warn("omni_normtest is not valid with less than 8 observations; %i "


In [11]:
# Let's detect ill-conditioned matrices.

# Slightly perturbed matrix
A_ill = np.array([[1, 1],
                  [1, 1.00001]])
cond_number = np.linalg.cond(A_ill)
print("Condition number (ill):", cond_number)

# Solving system
b = np.array([2, 2.00001])
x = np.linalg.solve(A_ill, b)  # Small changes in b = large changes in x.

# If cond_number >> 10**5, numerical instability is significant. Small errors in b or A amplify drastically.

Condition number (ill): 400002.0000028382


Notes:
1. Avoid explicit inverses for large systems; always prefer solve or SVD-based pseudoinverses.
2. Ill-conditioned matrices occur with nearly linearly dependent columns; check cond(A) before solving.
3. Floating-point errors: large/small numbers can lead to rounding errors; monitor singular values.
4. Regression & linear systems are deeply connected: solving XtX Beta = Xty is a linear system.

In [7]:
# The Role of Residual

X = np.array([
    [1,2],
    [1,3],
    [1,4],
    [1,5],
    [1,6],
])

# Create a target y.
y = np.array([1,2,3,4,5])

# Compute regression coefficients using least squares.
beta = np.linalg.lstsq(X, y, rcond=None) [0]

# Compute residuals.
r = y - X @ beta

# Check orthogonality.
ortho_check = X.T @ r
print("X.T @ r (should be near zero):", ortho_check)

X.T @ r (should be near zero): [4.88498131e-15 1.55431223e-14]
